# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [1]:
#!pip install librosa
#!pip install panns-inference

In [2]:
#!pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

In [3]:
#!pip install datasets

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [5]:
from huggingface_hub import HfApi
api = HfApi()
print(api.dataset_info("ashraq/esc50"))

C:\Users\sol_v\OneDrive\Escritorio\IRON_AI\langchain-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetInfo(id='ashraq/esc50', author='ashraq', card_data=None, citation=None, created_at=datetime.datetime(2022, 9, 24, 19, 51, 49, tzinfo=datetime.timezone.utc), description='https://github.com/karolpiczak/ESC-50\nThe dataset is available under the terms of the Creative Commons Attribution Non-Commercial license.\nK. J. Piczak. ESC: Dataset for Environmental Sound Classification. Proceedings of the 23rd Annual ACM Conference on Multimedia, Brisbane, Australia, 2015.\n[DOI: http://dx.doi.org/10.1145/2733373.2806390]\n', disabled=False, downloads=2935, downloads_all_time=None, gated=False, last_modified=datetime.datetime(2023, 1, 7, 8, 35, 28, tzinfo=datetime.timezone.utc), likes=38, main_size=None, paperswithcode_id=None, private=False, resource_group=None, sha='e3e2a63ffff66b9a9735524551e3818e96af03ee', siblings=[RepoSibling(rfilename='.gitattributes', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='README.md', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='dat

In [6]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

In [7]:
#import sys
#!{sys.executable} -m pip install torchcodec torch

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [8]:
import datasets
print(datasets.__version__)

5.0.1


In [9]:
#import sys
#!{sys.executable} -m pip install soundfile

In [10]:
# 1. Load the dataset (fresh after restart)
from datasets import load_dataset, Audio
data = load_dataset("ashraq/esc50", split="train")

# 2. Turn off auto-decoding so torchcodec is never invoked
data = data.cast_column("audio", Audio(decode=False))

# 3. Now manually decode with soundfile instead of the old audios[:3] line
import soundfile as sf
import io

audios = data["audio"][:3]

for a in audios:
    array, sampling_rate = sf.read(io.BytesIO(a["bytes"]))
    print("shape:", array.shape, "| sample rate:", sampling_rate)

Repo card metadata block was not found. Setting CardData to empty.


shape: (220500,) | sample rate: 44100
shape: (220500,) | sample rate: 44100
shape: (220500,) | sample rate: 44100


In [11]:
from datasets import Audio
import soundfile as sf
import io

# stop datasets from auto-decoding audio (this is what needs torchcodec)
data = data.cast_column("audio", Audio(decode=False))

audios = data["audio"][:3]

for a in audios:
    array, sampling_rate = sf.read(io.BytesIO(a["bytes"]))
    print("shape:", array.shape, "| sample rate:", sampling_rate)

shape: (220500,) | sample rate: 44100
shape: (220500,) | sample rate: 44100
shape: (220500,) | sample rate: 44100


We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [12]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
#audios = np.array([a["array"] for a in data["audio"]])

import numpy as np
import soundfile as sf
import io

# select only the audio data from the dataset and store in a numpy array
audios = np.array([sf.read(io.BytesIO(a["bytes"]))[0] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [13]:
#import sys
#!{sys.executable} -m pip install panns-inference

In [14]:
#import sys
#!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu

In [17]:
import os
import requests
from tqdm.auto import tqdm

checkpoint_dir = os.path.expanduser("~/panns_data")
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "Cnn14_mAP=0.431.pth")

url = "https://zenodo.org/records/3987831/files/Cnn14_mAP%3D0.431.pth?download=1"

# remove any partial/corrupt leftover from a failed attempt
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)

with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(checkpoint_path, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True, desc="Downloading"
    ) as pbar:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            pbar.update(len(chunk))

size_mb = os.path.getsize(checkpoint_path) / 1e6
print(f"Saved to: {checkpoint_path}")
print(f"Size: {size_mb:.1f} MB")
assert size_mb > 300, "File too small — download likely failed or returned an error page"
print("Looks good.")

Downloading: 100%|██████████| 327M/327M [11:00<00:00, 496kB/s]    

Saved to: C:\Users\sol_v/panns_data\Cnn14_mAP=0.431.pth
Size: 327.4 MB
Looks good.


In [15]:
import os
import urllib.request

panns_data_dir = os.path.expanduser("~/panns_data")
os.makedirs(panns_data_dir, exist_ok=True)

url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"
dest = os.path.join(panns_data_dir, "class_labels_indices.csv")

urllib.request.urlretrieve(url, dest)
print("Downloaded to:", dest)

Downloaded to: C:\Users\sol_v/panns_data\class_labels_indices.csv


In [18]:
from panns_inference import AudioTagging

# load the default model into the gpu.
model = AudioTagging(checkpoint_path=None, device='cpu') # change device to cpu if a gpu is not available

Checkpoint path: C:\Users\sol_v/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [19]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [20]:
#import sys
#!{sys.executable} -m pip install pinecone

In [21]:

import os
from dotenv import load_dotenv
from pinecone import Pinecone

# load variables from .env in the same folder as this notebook
load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")

if pinecone_api_key is None:
    raise ValueError("PINECONE_API_KEY not found — check that your .env file exists and is in the same folder as this notebook")

pc = Pinecone(api_key=pinecone_api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [22]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [23]:
index_name = "audio-search-demo"

In [24]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=2048, total_vector_count=0, metric='cosine', namespaces=0)

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [25]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

100%|██████████| 32/32 [11:02<00:00, 20.69s/it]


DescribeIndexStatsResponse(dimension=2048, total_vector_count=2000, metric='cosine', namespaces=1)

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [27]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["bytes"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [29]:
import numpy as np
import soundfile as sf
import io

# if query_audio is still raw bytes, decode it to a waveform array
if isinstance(query_audio, (bytes, bytearray)):
    query_audio, sr = sf.read(io.BytesIO(query_audio))

# PANNs expects float32
query_audio = np.asarray(query_audio, dtype=np.float32)
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [30]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

QueryResponse(matches=[ScoredVector(id='400', score=0.99963063, values=[]), ScoredVector(id='1667', score=0.842086911, values=[]), ScoredVector(id='1666', score=0.831266105, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 06 Sep 2026 19:26:11 GMT', 'content-type': 'application/json', 'content-length': '201', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '32', 'x-pinecone-request-latency-ms': '50', 'x-envoy-upstream-service-time': '43', 'x-pinecone-response-duration-ms': '52', 'grpc-status': '0', 'server': 'envoy'}))

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [32]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["bytes"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [34]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["bytes"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["bytes"]
        display(Audio(a, rate=44100))

In [35]:
find_similar_audios(1642)

Query Audio:


Result:


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [36]:
find_similar_audios(452)

Query Audio:


Result:


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [37]:
#!wget https://storage.googleapis.com/audioset/miaow_16k.wav

We can load the audio into a Numpy array as follows:

In [38]:
import librosa

a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [41]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["bytes"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [42]:
pc.delete_index(index_name)